# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

*All entities (record sets, fields, columns, etc.) are referenced by their `@id` fields throughout this notebook.*

In [ ]:
# Ensure `mlcroissant` is installed (run this cell if needed)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a metadata object, not a dictionary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore record sets, fields, and their `@id`s.

### List available record sets

In [ ]:
# List all available record sets by @id and print their structures
print("Available record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs.id} - name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    # List fields in this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name}")
    print("")

Based on the above listing, choose a record set and its fields for analysis. You can refer to the IDs (the `@id` values printed above) in subsequent cells.

For demonstration, we will select the main protein record set, which is typically named or described accordingly (adjust the `record_set_id` below if the main table differs).

You can further inspect example records within a given record set using the code below (replace `<record_set_id>` with a valid record set @id such as `'cr:RecordSet_1'`).

In [ ]:
# Example: preview records from a record set by @id
record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'protein' in (rs.name or '').lower() or 'main' in (rs.name or '').lower():
        record_set_id = rs.id
        break
if not record_set_id:
    # Fallback to the first record set
    record_set_id = dataset.metadata.record_sets[0].id
print(f"Previewing records from record set: {record_set_id}\n")
for i, rec in enumerate(dataset.records(record_set=record_set_id)):
    print(json.dumps(rec, indent=2))
    if i >= 2:
        break

## 3. Data Extraction
Load the main data tables into Pandas DataFrames. Use record set and field `@id`s for selection.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"Loaded data for record set @id: {rs_id} - shape: {df.shape}")

# Show columns of the primary record set
print(f"\nColumns for primary record set ('{record_set_id}'):")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll carry out some common analysis steps:
- Filter numeric fields (e.g., by peptide count, molecular weight, etc.).
- Normalize a numeric column.
- Group by a protein attribute.

Below, we'll attempt to pick an appropriate field for demonstration. Adjust the variable assignments below to select relevant fields for the EDA.

In [ ]:
# Automatically choose a numeric field for analysis
import numpy as np
df = dataframes[record_set_id]

# Try to choose a likely numeric field
numeric_field = None
group_field = None
for col in df.columns:
    # Try to detect numeric columns
    if np.issubdtype(df[col].dropna().astype(str).str.replace(',', '').str.replace(' ', '').astype(float, errors='ignore').dtype, np.number):
        numeric_field = col
        break
# Choose a grouping field (often categorical)
for col in df.columns:
    # Exclude the numeric field
    if col != numeric_field and df[col].nunique() < df.shape[0]/2:
        group_field = col
        break
        
if numeric_field is not None:
    try:
        df[numeric_field] = pd.to_numeric(df[numeric_field].astype(str).str.replace(',', '').str.replace(' ', ''), errors='coerce')
    except:
        pass
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]}")
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First few normalized records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group if suitable
    if group_field is not None and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by {group_field}, showing mean {numeric_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and relationship with a categorical variable (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# Boxplot grouped by field, if both variables available
if group_field and numeric_field and group_field in df.columns:
    plt.figure(figsize=(12,4))
    top_cats = df[group_field].value_counts().nlargest(10).index
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_cats)])
    plt.title(f"{numeric_field} by {group_field} (top 10 categories)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset defined by Croissant schema.

- We loaded the dataset directly from its Croissant schema URL.
- We identified available record sets and fields via their `@id`s for reproducible access.
- Basic data extraction and exploratory analysis were performed, including filtering, normalization, grouping, and visualizations.

Further domain-specific analysis can be undertaken using this structured approach, referencing the exact `@id` fields defined in the schema for reliable scientific data processing.